# CAFA5 Savio Tuned Training Results Analysis

Objective: analyze outputs from `scripts/savio_train_full_graph_tuned.sh` without retraining.

This notebook is tuned-run specific. It reads the newest `full_graph_tuned_*` run by default and reports:

- final and best-checkpoint metrics by GO aspect
- whether early stopping triggered
- `weighted_bce` / `pos_weight` diagnostics
- LR scheduler behavior
- validation/test learning curves
- optional comparison against the newest baseline `full_graph_pyg_mtf20_*` run

To analyze a specific run, set `CAFA5_TUNED_ANALYSIS_RUN_DIR` before running the setup cell.

In [ ]:
# Setup and run discovery
from __future__ import annotations

import json
import os
import re
from pathlib import Path
from typing import Any

try:
    import pandas as pd
except ImportError as exc:
    raise ImportError('pandas is required for this analysis notebook') from exc

try:
    import matplotlib.pyplot as plt
except ImportError as exc:
    raise ImportError('matplotlib is required for plotting in this notebook') from exc

from IPython.display import Markdown, display


def find_repo_root(start: Path) -> tuple[Path, str]:
    for candidate in (start, *start.parents):
        if (candidate / '.git').exists():
            return candidate, 'cwd_search'
    return start, 'cwd_fallback'


def resolve_repo_root() -> tuple[Path, str]:
    configured = os.environ.get('CAFA5_REPO_ROOT') or os.environ.get('REPO_ROOT')
    if configured:
        return Path(configured).expanduser().resolve(), 'env:repo_root'
    return find_repo_root(Path.cwd())


def newest_run_dir(training_runs_dir: Path, pattern: str) -> Path:
    candidates = [path for path in training_runs_dir.glob(pattern) if path.is_dir()]
    if not candidates:
        raise FileNotFoundError(f'No run directories matched {training_runs_dir / pattern}')
    return max(candidates, key=lambda path: path.stat().st_mtime)


def resolve_run_dir(env_key: str, training_runs_dir: Path, pattern: str) -> Path:
    configured = os.environ.get(env_key)
    if configured:
        return Path(configured).expanduser().resolve()
    return newest_run_dir(training_runs_dir, pattern).resolve()


REPO_ROOT, REPO_ROOT_SOURCE = resolve_repo_root()
SERVER_USER = os.environ.get('USER', 'bensonli')
RUN_ROOT = Path(os.environ.get('CAFA5_RUN_ROOT', f'/global/scratch/users/{SERVER_USER}/cafa5_outputs')).expanduser()
GRAPH_CACHE_DIR = RUN_ROOT / 'graph_cache'
TRAINING_RUNS_DIR = GRAPH_CACHE_DIR / 'training_runs'
TUNED_RUN_GLOB = os.environ.get('CAFA5_TUNED_ANALYSIS_RUN_GLOB', 'full_graph_tuned_*')
BASELINE_RUN_GLOB = os.environ.get('CAFA5_BASELINE_ANALYSIS_RUN_GLOB', 'full_graph_pyg_mtf20_*')

RUN_DIR = resolve_run_dir('CAFA5_TUNED_ANALYSIS_RUN_DIR', TRAINING_RUNS_DIR, TUNED_RUN_GLOB)
FIGURE_DIR = RUN_DIR / 'analysis_figures_tuned'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

config = {
    'repo_root': str(REPO_ROOT),
    'repo_root_source': REPO_ROOT_SOURCE,
    'run_root': str(RUN_ROOT),
    'training_runs_dir': str(TRAINING_RUNS_DIR),
    'tuned_run_glob': TUNED_RUN_GLOB,
    'run_dir': str(RUN_DIR),
    'figure_dir': str(FIGURE_DIR),
}
print(json.dumps(config, indent=2))

if not RUN_DIR.exists():
    raise FileNotFoundError(f'Missing tuned run directory: {RUN_DIR}')

## Load Run Artifacts

The tuned batch script writes a top-level `results_summary.json` plus one subdirectory per aspect (`bpo/`, `cco/`, `mfo/`). This section loads the run config, Git metadata, final summaries, and per-epoch histories.

In [ ]:
# Load top-level config and result summary.
def read_json_if_exists(path: Path, default: Any = None) -> Any:
    if not path.exists():
        return default
    return json.loads(path.read_text(encoding='utf-8'))


def read_text_if_exists(path: Path, default: str = '') -> str:
    return path.read_text(encoding='utf-8', errors='replace').strip() if path.exists() else default


def load_results_rows(run_dir: Path) -> list[dict[str, Any]]:
    rows = read_json_if_exists(run_dir / 'results_summary.json', default=[])
    if rows:
        return rows
    tsv_path = run_dir / 'results_summary.tsv'
    if tsv_path.exists():
        return pd.read_csv(tsv_path, sep='\t').to_dict('records')
    raise FileNotFoundError(f'No results summary found in {run_dir}')


run_config = read_json_if_exists(RUN_DIR / 'run_config.json', default={})
results_rows = load_results_rows(RUN_DIR)
results_df = pd.DataFrame(results_rows)

metadata = {
    'git_branch': read_text_if_exists(RUN_DIR / 'git_branch.txt'),
    'git_commit': read_text_if_exists(RUN_DIR / 'git_commit.txt'),
    'git_status': read_text_if_exists(RUN_DIR / 'git_status.txt'),
    'run_config_exists': (RUN_DIR / 'run_config.json').exists(),
    'results_summary_json_exists': (RUN_DIR / 'results_summary.json').exists(),
    'results_summary_tsv_exists': (RUN_DIR / 'results_summary.tsv').exists(),
    'driver_log_exists': (RUN_DIR / 'driver.log').exists(),
}
print(json.dumps(metadata, indent=2))
display(results_df)

In [ ]:
# Load per-aspect run_result.json and summary.json histories.
def aspect_dirs(run_dir: Path) -> list[Path]:
    return sorted(path for path in run_dir.iterdir() if path.is_dir() and (path / 'run_result.json').exists())


def flatten_epoch_history(aspect: str, summary: dict[str, Any]) -> list[dict[str, Any]]:
    rows = []
    for record in summary.get('history', []) or []:
        for split_name in ('train', 'val', 'test'):
            metrics = record.get(split_name, {}) or {}
            row = {
                'aspect': aspect,
                'split': split_name,
                'epoch': record.get('epoch'),
                'epoch_seconds': record.get('epoch_seconds'),
                'total_elapsed_seconds': record.get('total_elapsed_seconds'),
                'average_epoch_seconds': record.get('average_epoch_seconds'),
                'lr': record.get('lr'),
                'checkpoint_metric_name': record.get('checkpoint_metric_name'),
                'checkpoint_metric_value': record.get('checkpoint_metric_value'),
            }
            for key in (
                'loss',
                'micro_precision',
                'micro_recall',
                'micro_f1',
                'macro_precision',
                'macro_recall',
                'macro_f1',
                'macro_f1_positive_labels',
                'fmax',
                'fmax_threshold',
                'fmax_precision',
                'fmax_recall',
                'graphs',
            ):
                row[key] = metrics.get(key)
            rows.append(row)
    return rows


run_result_rows = []
epoch_rows = []
summary_rows = []
pos_weight_rows = []
summary_by_aspect = {}

for aspect_dir in aspect_dirs(RUN_DIR):
    run_result = read_json_if_exists(aspect_dir / 'run_result.json', default={})
    summary = read_json_if_exists(aspect_dir / 'summary.json', default={})
    if not run_result:
        continue
    aspect = str(run_result.get('aspect') or aspect_dir.name.upper())
    summary_by_aspect[aspect] = summary
    run_result_rows.append({
        'aspect': aspect,
        'status': run_result.get('status'),
        'status_code': run_result.get('status_code'),
        'gpu_id': run_result.get('gpu_id'),
        'started_at': run_result.get('started_at'),
        'finished_at': run_result.get('finished_at'),
        'checkpoint_dir': run_result.get('checkpoint_dir'),
        'summary_path': run_result.get('summary_path'),
        'best_checkpoint_path': run_result.get('best_checkpoint_path'),
        'best_val_loss': run_result.get('best_val_loss'),
        'best_checkpoint_metric_name': run_result.get('best_checkpoint_metric_name'),
        'best_checkpoint_metric': run_result.get('best_checkpoint_metric'),
        'best_epoch': run_result.get('best_epoch'),
        'stopped_early': run_result.get('stopped_early'),
        'loss_function': run_result.get('loss_function'),
        'lr_scheduler': run_result.get('lr_scheduler'),
    })
    summary_rows.append({
        'aspect': aspect,
        'epochs_requested': summary.get('epochs'),
        'epochs_completed': summary.get('epochs_completed'),
        'best_epoch': summary.get('best_epoch'),
        'checkpoint_metric': summary.get('checkpoint_metric'),
        'best_checkpoint_metric': summary.get('best_checkpoint_metric'),
        'best_checkpoint_metric_mode': summary.get('best_checkpoint_metric_mode'),
        'stopped_early': summary.get('stopped_early'),
        'early_stopping_reason': summary.get('early_stopping_reason'),
        'loss_function': summary.get('loss_function'),
        'lr_scheduler': summary.get('lr_scheduler'),
        'batch_size': summary.get('batch_size'),
        'hidden_dim': summary.get('hidden_dim'),
        'dropout': summary.get('dropout'),
        'lr': summary.get('lr'),
        'weight_decay': summary.get('weight_decay'),
    })
    loss_config = summary.get('loss_config') or {}
    pos_summary = loss_config.get('pos_weight_summary') or {}
    if pos_summary:
        pos_weight_rows.append({'aspect': aspect, **loss_config, **pos_summary})
    epoch_rows.extend(flatten_epoch_history(aspect, summary))

run_result_df = pd.DataFrame(run_result_rows)
epoch_df = pd.DataFrame(epoch_rows)
summary_df = pd.DataFrame(summary_rows)
pos_weight_df = pd.DataFrame(pos_weight_rows)

display(run_result_df)
display(summary_df)
print('epoch rows =', len(epoch_df))

## Final Metrics

Use this table for a quick status check. It shows the final epoch metrics produced by `results_summary.*`. The tuned run also records the selected checkpoint metric and best epoch.

In [ ]:
# Final metric table focused on validation/test performance and tuning controls.
metric_cols = [
    'aspect',
    'status',
    'status_code',
    'final_epoch',
    'best_epoch',
    'best_checkpoint_metric_name',
    'best_checkpoint_metric',
    'stopped_early',
    'loss_function',
    'lr_scheduler',
    'final_lr',
    'best_val_loss',
    'val_loss',
    'val_micro_f1',
    'val_macro_f1',
    'val_fmax',
    'val_fmax_threshold',
    'test_loss',
    'test_micro_f1',
    'test_macro_f1',
    'test_fmax',
    'test_fmax_threshold',
    'checkpoint_dir',
]
available_cols = [col for col in metric_cols if col in results_df.columns]
final_metrics_df = results_df[available_cols].copy()
for col in final_metrics_df.columns:
    if any(token in col for token in ('loss', 'f1', 'fmax', 'threshold', 'metric', 'lr')):
        final_metrics_df[col] = pd.to_numeric(final_metrics_df[col], errors='ignore')

display(final_metrics_df.sort_values(['status', 'aspect']))
summary_out = FIGURE_DIR / 'final_metrics_tuned.tsv'
final_metrics_df.to_csv(summary_out, sep='\t', index=False)
print('wrote', summary_out)

## Best Checkpoint vs Final Epoch

The tuned script selects `best.pt` using `CHECKPOINT_METRIC`, which defaults to `val_fmax`. This section compares metrics at the best epoch against the final epoch, which is important when early stopping did not trigger but validation Fmax peaked earlier.

In [ ]:
# Compare best-checkpoint epoch to final epoch for each aspect.
def metric_at_epoch(epoch_df: pd.DataFrame, aspect: str, epoch: int | float | None, split: str, metric: str) -> float | None:
    if epoch is None or pd.isna(epoch):
        return None
    rows = epoch_df[(epoch_df['aspect'] == aspect) & (epoch_df['split'] == split) & (epoch_df['epoch'] == int(epoch))]
    if rows.empty or metric not in rows.columns:
        return None
    return rows.iloc[0][metric]


comparison_rows = []
for _, row in summary_df.iterrows():
    aspect = row['aspect']
    best_epoch = row.get('best_epoch')
    aspect_epochs = epoch_df[epoch_df['aspect'] == aspect]
    final_epoch = aspect_epochs['epoch'].max() if not aspect_epochs.empty else None
    payload = {
        'aspect': aspect,
        'status': run_result_df.set_index('aspect').get('status', pd.Series()).get(aspect),
        'checkpoint_metric': row.get('checkpoint_metric'),
        'best_epoch': best_epoch,
        'final_epoch': final_epoch,
        'stopped_early': row.get('stopped_early'),
    }
    for split_name in ('val', 'test'):
        for metric in ('loss', 'micro_f1', 'macro_f1', 'fmax', 'fmax_threshold'):
            payload[f'best_{split_name}_{metric}'] = metric_at_epoch(epoch_df, aspect, best_epoch, split_name, metric)
            payload[f'final_{split_name}_{metric}'] = metric_at_epoch(epoch_df, aspect, final_epoch, split_name, metric)
        payload[f'delta_{split_name}_fmax'] = (
            payload.get(f'best_{split_name}_fmax') - payload.get(f'final_{split_name}_fmax')
            if payload.get(f'best_{split_name}_fmax') is not None and payload.get(f'final_{split_name}_fmax') is not None
            else None
        )
    comparison_rows.append(payload)

best_vs_final_df = pd.DataFrame(comparison_rows)
display(best_vs_final_df)
best_vs_final_out = FIGURE_DIR / 'best_vs_final_tuned.tsv'
best_vs_final_df.to_csv(best_vs_final_out, sep='\t', index=False)
print('wrote', best_vs_final_out)

## Weighted BCE Diagnostics

The tuned run defaults to `weighted_bce` with capped and softened positive weights. This table checks whether the run actually used class weighting and how strong the resulting weights were per aspect.

In [ ]:
# Inspect weighted BCE / pos_weight settings.
if pos_weight_df.empty:
    display(Markdown('No `pos_weight_summary` found. The run may have used plain BCE or an older trainer.'))
else:
    preferred_cols = [
        'aspect',
        'loss_function',
        'pos_weight_power',
        'max_pos_weight',
        'label_count',
        'weighted_label_count',
        'min',
        'mean',
        'max',
    ]
    display(pos_weight_df[[col for col in preferred_cols if col in pos_weight_df.columns]])

## Learning Curves

These plots diagnose underfitting, overfitting, and plateau behavior. For the tuned run, focus on validation/test `fmax` and whether the best epoch differs from the final epoch.

In [ ]:
# Plot loss, micro-F1, macro-F1, and Fmax by epoch.
if epoch_df.empty:
    raise ValueError('epoch_df is empty. No per-epoch summary.json histories were found.')

aspects = list(epoch_df['aspect'].dropna().unique())
metrics_to_plot = ['loss', 'micro_f1', 'macro_f1', 'fmax']
fig, axes = plt.subplots(
    len(metrics_to_plot),
    len(aspects),
    figsize=(5 * max(1, len(aspects)), 3.2 * len(metrics_to_plot)),
    squeeze=False,
)
for col_index, aspect in enumerate(aspects):
    aspect_df = epoch_df[epoch_df['aspect'] == aspect]
    best_epoch = summary_df.set_index('aspect').get('best_epoch', pd.Series()).get(aspect)
    for row_index, metric in enumerate(metrics_to_plot):
        ax = axes[row_index][col_index]
        for split_name in ('train', 'val', 'test'):
            split_df = aspect_df[aspect_df['split'] == split_name].sort_values('epoch')
            if split_df.empty or metric not in split_df.columns:
                continue
            ax.plot(split_df['epoch'], pd.to_numeric(split_df[metric], errors='coerce'), marker='o', label=split_name)
        if pd.notna(best_epoch):
            ax.axvline(int(best_epoch), linestyle='--', color='black', alpha=0.35, label='best epoch')
        ax.set_title(f'{aspect} {metric}')
        ax.set_xlabel('epoch')
        ax.grid(True, alpha=0.3)
        if col_index == 0:
            ax.set_ylabel(metric)
        if row_index == 0 and col_index == len(aspects) - 1:
            ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
figure_path = FIGURE_DIR / 'learning_curves_tuned.png'
plt.savefig(figure_path, dpi=160, bbox_inches='tight')
plt.show()
print('wrote', figure_path)

## Tuning Controls Over Time

This section checks the controls added for the tuned script: learning rate, checkpoint metric, and Fmax threshold. It helps confirm whether the scheduler fired and whether threshold selection moved materially across epochs.

In [ ]:
# Plot LR and checkpoint metric by epoch.
unique_epoch_rows = epoch_df[epoch_df['split'] == 'val'].copy()
unique_epoch_rows['lr'] = pd.to_numeric(unique_epoch_rows['lr'], errors='coerce')
unique_epoch_rows['checkpoint_metric_value'] = pd.to_numeric(unique_epoch_rows['checkpoint_metric_value'], errors='coerce')

fig, axes = plt.subplots(1, 2, figsize=(12, 4), squeeze=False)
for aspect, group in unique_epoch_rows.groupby('aspect'):
    group = group.sort_values('epoch')
    axes[0][0].plot(group['epoch'], group['lr'], marker='o', label=aspect)
    axes[0][1].plot(group['epoch'], group['checkpoint_metric_value'], marker='o', label=aspect)
axes[0][0].set_title('Learning rate by epoch')
axes[0][0].set_xlabel('epoch')
axes[0][0].set_ylabel('lr')
axes[0][0].grid(True, alpha=0.3)
axes[0][1].set_title('Checkpoint metric by epoch')
axes[0][1].set_xlabel('epoch')
axes[0][1].set_ylabel('metric value')
axes[0][1].grid(True, alpha=0.3)
axes[0][1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
figure_path = FIGURE_DIR / 'lr_and_checkpoint_metric_tuned.png'
plt.savefig(figure_path, dpi=160, bbox_inches='tight')
plt.show()
print('wrote', figure_path)

# Plot validation/test Fmax thresholds.
threshold_df = epoch_df[epoch_df['split'].isin(['val', 'test'])].copy()
threshold_df['fmax_threshold'] = pd.to_numeric(threshold_df['fmax_threshold'], errors='coerce')
fig, ax = plt.subplots(figsize=(8, 4))
for (aspect, split_name), group in threshold_df.groupby(['aspect', 'split']):
    group = group.sort_values('epoch')
    ax.plot(group['epoch'], group['fmax_threshold'], marker='o', label=f'{aspect} {split_name}')
ax.set_title('Fmax threshold by epoch')
ax.set_xlabel('epoch')
ax.set_ylabel('threshold')
ax.set_ylim(-0.02, 1.02)
ax.grid(True, alpha=0.3)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
figure_path = FIGURE_DIR / 'fmax_thresholds_tuned.png'
plt.savefig(figure_path, dpi=160, bbox_inches='tight')
plt.show()
print('wrote', figure_path)

## Runtime and Progress Logs

Use this section to spot failures, early exits, and dataloader stalls. `BPO` was previously the highest-risk aspect, so check its status and log tail first if it fails again.

In [ ]:
# Runtime summary and lightweight train.log parsing.
runtime_df = run_result_df.copy()
for col in ('started_at', 'finished_at'):
    runtime_df[col] = pd.to_datetime(runtime_df[col], errors='coerce')
runtime_df['duration_minutes'] = (runtime_df['finished_at'] - runtime_df['started_at']).dt.total_seconds() / 60
display(runtime_df[['aspect', 'status', 'status_code', 'gpu_id', 'duration_minutes', 'stopped_early', 'checkpoint_dir']])

fig, ax = plt.subplots(figsize=(7, 4))
runtime_df.plot(kind='bar', x='aspect', y='duration_minutes', ax=ax, legend=False)
ax.set_title('Training duration by aspect')
ax.set_xlabel('aspect')
ax.set_ylabel('minutes')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
figure_path = FIGURE_DIR / 'runtime_by_aspect_tuned.png'
plt.savefig(figure_path, dpi=160, bbox_inches='tight')
plt.show()
print('wrote', figure_path)

progress_pattern = re.compile(
    r"\[progress\]\s+epoch=(?P<epoch>\d+)\s+(?P<label>.+?)\s+batch=(?P<batch>\d+)/(?:\?|(?P<total_batches>\d+))\s+graphs=(?P<graphs>\d+)\s+loss=(?P<loss>[-+0-9.eE]+)"
)
progress_rows = []
for aspect_dir in aspect_dirs(RUN_DIR):
    aspect = aspect_dir.name.upper()
    log_path = aspect_dir / 'train.log'
    if not log_path.exists():
        continue
    for line in log_path.read_text(encoding='utf-8', errors='replace').splitlines():
        match = progress_pattern.search(line)
        if not match:
            continue
        payload = match.groupdict()
        label_parts = payload['label'].split()
        split_name = label_parts[-1] if label_parts else None
        progress_rows.append({
            'aspect': aspect,
            'epoch': int(payload['epoch']),
            'split': split_name,
            'batch': int(payload['batch']),
            'total_batches': int(payload['total_batches']) if payload.get('total_batches') else None,
            'graphs': int(payload['graphs']),
            'loss': float(payload['loss']),
        })

progress_df = pd.DataFrame(progress_rows)
if progress_df.empty:
    display(Markdown('No progress rows parsed from `train.log`.'))
else:
    display(progress_df.groupby(['aspect', 'epoch', 'split']).tail(1).sort_values(['aspect', 'epoch', 'split']).head(30))
    fig, ax = plt.subplots(figsize=(9, 4))
    for (aspect, split_name), group in progress_df.groupby(['aspect', 'split']):
        if split_name != 'train':
            continue
        group = group.sort_values(['epoch', 'batch'])
        x = range(len(group))
        ax.plot(x, group['loss'], label=f'{aspect} train')
    ax.set_title('Training progress loss from logs')
    ax.set_xlabel('logged progress point')
    ax.set_ylabel('running loss')
    ax.grid(True, alpha=0.3)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    figure_path = FIGURE_DIR / 'progress_running_loss_tuned.png'
    plt.savefig(figure_path, dpi=160, bbox_inches='tight')
    plt.show()
    print('wrote', figure_path)

## Optional Baseline Comparison

If an earlier baseline run exists under `full_graph_pyg_mtf20_*`, this section compares tuned results against it. Use `CAFA5_BASELINE_ANALYSIS_RUN_DIR` to select a specific baseline run.

In [ ]:
# Compare tuned final metrics to the newest baseline run if available.
def maybe_load_baseline_run() -> tuple[Path | None, pd.DataFrame | None]:
    configured = os.environ.get('CAFA5_BASELINE_ANALYSIS_RUN_DIR')
    try:
        baseline_dir = Path(configured).expanduser().resolve() if configured else newest_run_dir(TRAINING_RUNS_DIR, BASELINE_RUN_GLOB).resolve()
    except FileNotFoundError:
        return None, None
    if baseline_dir == RUN_DIR:
        return None, None
    try:
        return baseline_dir, pd.DataFrame(load_results_rows(baseline_dir))
    except FileNotFoundError:
        return baseline_dir, None


baseline_dir, baseline_df = maybe_load_baseline_run()
if baseline_df is None or baseline_df.empty:
    display(Markdown('No baseline run found or baseline has no summary.'))
else:
    print('baseline_dir =', baseline_dir)
    tuned = final_metrics_df.copy()
    baseline = baseline_df.copy()
    compare_cols = ['aspect', 'status', 'final_epoch', 'val_fmax', 'test_fmax', 'val_micro_f1', 'test_micro_f1', 'val_macro_f1', 'test_macro_f1']
    tuned_small = tuned[[col for col in compare_cols if col in tuned.columns]].add_prefix('tuned_')
    tuned_small = tuned_small.rename(columns={'tuned_aspect': 'aspect'})
    baseline_small = baseline[[col for col in compare_cols if col in baseline.columns]].add_prefix('baseline_')
    baseline_small = baseline_small.rename(columns={'baseline_aspect': 'aspect'})
    comparison = tuned_small.merge(baseline_small, on='aspect', how='outer')
    for metric in ('val_fmax', 'test_fmax', 'val_micro_f1', 'test_micro_f1', 'val_macro_f1', 'test_macro_f1'):
        tuned_col = f'tuned_{metric}'
        base_col = f'baseline_{metric}'
        if tuned_col in comparison.columns and base_col in comparison.columns:
            comparison[f'delta_{metric}'] = pd.to_numeric(comparison[tuned_col], errors='coerce') - pd.to_numeric(comparison[base_col], errors='coerce')
    display(comparison)
    comparison_out = FIGURE_DIR / 'baseline_comparison_tuned.tsv'
    comparison.to_csv(comparison_out, sep='\t', index=False)
    print('wrote', comparison_out)

## Interpretation Notes

Run this final cell after the tables and plots are generated. It creates a short checklist of the most important tuned-run outcomes.

In [ ]:
# Generate a compact interpretation checklist.
notes = []
if 'status' in final_metrics_df.columns:
    failed = final_metrics_df[final_metrics_df['status'] != 'success']
    if not failed.empty:
        notes.append('Some aspects did not finish successfully: ' + ', '.join(failed['aspect'].astype(str)))
    else:
        notes.append('All aspects finished successfully.')

if 'best_epoch' in best_vs_final_df.columns:
    changed = best_vs_final_df[best_vs_final_df['best_epoch'] != best_vs_final_df['final_epoch']]
    if not changed.empty:
        notes.append('Best checkpoint differs from final epoch for: ' + ', '.join(changed['aspect'].astype(str)))
    else:
        notes.append('Best checkpoint epoch equals final epoch for all completed aspects.')

if 'test_fmax' in final_metrics_df.columns:
    ranked = final_metrics_df.copy()
    ranked['test_fmax'] = pd.to_numeric(ranked['test_fmax'], errors='coerce')
    ranked = ranked.sort_values('test_fmax', ascending=False)
    if not ranked.empty:
        best_row = ranked.iloc[0]
        notes.append(f"Best final test Fmax: {best_row.get('aspect')} = {best_row.get('test_fmax'):.4f}")

if not pos_weight_df.empty:
    notes.append('Weighted BCE was active and pos_weight summaries were recorded.')
else:
    notes.append('No pos_weight summary found; verify LOSS_FUNCTION if class weighting was expected.')

notes.append('Next formal-evaluation step: export prediction TSVs and run cafaeval for IA-weighted CAFA-style wFmax.')

display(Markdown('\n'.join(f'- {note}' for note in notes)))

## Files Written

This notebook writes reusable analysis outputs under `RUN_DIR/analysis_figures_tuned/`:

- `final_metrics_tuned.tsv`
- `best_vs_final_tuned.tsv`
- `learning_curves_tuned.png`
- `lr_and_checkpoint_metric_tuned.png`
- `fmax_thresholds_tuned.png`
- `runtime_by_aspect_tuned.png`
- `progress_running_loss_tuned.png` when progress logs are available
- `baseline_comparison_tuned.tsv` when a baseline run is available